# BERT Fine-tuned POS Tagger


In [ ]:
!pip install seqeval evaluate datasets transformers

In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*seems not to be NE tag.*")

In [3]:
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification

# Device setup
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
from datasets import load_dataset
dataset = load_dataset("eriktks/conll2003", revision="convert/parquet")


print(dataset)
print("\nFirst training example:")
print(dataset["train"][0])

# Accessing the feature names correctly
pos_feature = dataset["train"].features["pos_tags"]
if hasattr(pos_feature, 'feature'):
    label_names = pos_feature.feature.names
else:
    label_names = pos_feature.names

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

num_labels = len(label_names)


In [5]:
print(f"\nNumber of POS tags: {num_labels}")
print(f"Tags: {label_names}")


Number of POS tags: 47
Tags: ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']


## 2. Tokenization and Alignment


In [ ]:
model_checkpoint = "prajjwal1/bert-mini"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, max_length=128)

    labels = []
    for i, label in enumerate(examples["pos_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens mapped to None
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Other tokens in a word get -100
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply tokenization
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)


## 3. Metrics Definition

In [ ]:
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [
        [label_names[l] for l in label if l != -100] for label in labels
    ]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


## 4. Model Setup and Training


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model.to(device)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

args = TrainingArguments(
    output_dir="./bert-mini-pos",
    eval_strategy="epoch",
    learning_rate=0.00002,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=20,
    weight_decay=0.01,
    push_to_hub=False,
    logging_steps=100,
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## 5. Training

In [ ]:
trainer.train()


## 6. Evaluation and Inference

In [ ]:
test_results = trainer.evaluate(tokenized_datasets["test"])


In [15]:
print(test_results)

{'eval_loss': 0.2898291349411011, 'eval_precision': 0.9036360532604985, 'eval_recall': 0.8874514515633295, 'eval_f1': 0.8954706288678688, 'eval_accuracy': 0.92957898137181, 'eval_runtime': 5.5694, 'eval_samples_per_second': 619.991, 'eval_steps_per_second': 19.392, 'epoch': 20.0}


In [11]:
from transformers import pipeline

pos_pipeline = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple", device=0 if device != "cpu" else -1)

text = "Michael Scott is the regional manager of Dunder Mifflin Paper Company."
print(f"\nInput text: {text}")
results = pos_pipeline(text)
for res in results:
    print(f"Word: {res['word']:<15} | POS: {res['entity_group']}")



Input text: Michael Scott is the regional manager of Dunder Mifflin Paper Company.
Word: Michael Scott   | POS: NNP
Word: is              | POS: VBZ
Word: the             | POS: DT
Word: regional        | POS: JJ
Word: manager         | POS: NN
Word: of              | POS: IN
Word: Dunder Mifflin Paper Company | POS: NNP
Word: .               | POS: .


In [12]:
import pandas as pd

predictions_output = trainer.predict(tokenized_datasets["test"])
final_metrics = compute_metrics((predictions_output.predictions, predictions_output.label_ids))

metrics_df = pd.DataFrame([final_metrics])
display(metrics_df)

,precision,recall,f1,accuracy
0,0.903636,0.887451,0.895471,0.929579
